# From a retained Carvana page to a vehicle record

**Question:** where do a VIN, listing ID and asking price in our table come from?

Read one saved source, find the same vehicle in the normalized table, then
reproduce its asking-price value. Normal Run All reads local files and writes
nothing. The real example contains three browser records observed on
**September 8, 2026 at 01:44:11.591 UTC**; it is a historical sample.

**One row means one retailer listing seen at one observation time.**

| Term used below | Meaning in this lesson |
| --- | --- |
| VIN | The vehicle's 17-character identifier; it follows the physical vehicle. |
| Listing ID | Carvana's identifier for this listing/page. A later listing for the same VIN can have a different ID. |
| Capture | A saved observation of source content with its own timestamp. Reloading it preserves that time. |
| Asking price | The advertised offer in USD, before any claim about a transaction price. |

## Settings

`retained_path` selects the real three-record JSON file.
`normalized_retained` will hold the parser's table. The later `source_file`
is an **invented CSV**, used only in the optional validation exercise.
Changing either path selects another local input; it makes no request.

Follow **00 → 10 → 11 → 20 → 24 → 30**. Each notebook names its own selected
evidence; they are not all views of one population. The
[code walkthrough](../docs/code_walkthrough.md) maps the full workflow.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / "vehicle" / "src").is_dir():
    ROOT = ROOT / "vehicle"
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src" / "vehicle_tracker").is_dir(), "Launch from researchOS, vehicle/, or vehicle/notebooks/."
sys.path.insert(0, str(ROOT / "src"))
import vehicle_tracker

# Read-only inputs. The second file is used only in the optional example.
retained_path = ROOT / 'tests/fixtures/carvana_browser_sample_20260907.json'
source_file = ROOT / 'tests/fixtures/synthetic_listings.csv'


## 1. Inspect the source fields, then their normalized names

The first display expands the JSON records into columns. The second shows the
same records after parsing: column names, numeric values and listing IDs have
been made consistent. No extra observation or vehicle has been created.

Follow listing **4474057**, VIN **3FA6P0D94ER264351**, in both tables:

| Retained source | Normalized column | This record |
| --- | --- | --- |
| `vehicleIdentificationNumber` | `vin` | `3FA6P0D94ER264351` |
| Final number in `offers.url` | `listing_id` | `4474057` |
| `offers.price` | `asking_price_usd` | `14590` |

The source's `InStock` wording remains in `availability_native`.
A missing field stays missing; it is not replaced with zero. Neither this status
nor the asking price establishes a sale. The printed file path and observation
time tell you exactly which saved example produced these rows.


In [ ]:
import json
from vehicle_tracker.carvana import parse_capture
retained = json.loads(retained_path.read_text(encoding='utf-8'))
display(pd.json_normalize(retained['records']).head())
normalized_retained = parse_capture(retained)
display(normalized_retained[['listing_id', 'vin', 'asking_price_usd', 'availability_native', 'source_url']])
print('Retained observation:', retained['captured_at_utc'], retained_path)
print('Sample only; no population coverage or sale is established.')

### Try one real-row calculation

Locate VIN `3FA6P0D94ER264351` in both displayed tables. In a new scratch cell, use
`normalized_retained.loc[normalized_retained.vin.eq('3FA6P0D94ER264351'), 'asking_price_usd']`.
You should recover **14590**, matching `offers.price`. Dividing by 1,000 gives
**14.59 thousand USD**; this changes the unit, not the underlying price or evidence.
Next, choose another retained VIN and repeat the lookup. No request or save is needed.


## Optional learning example: validate an invented CSV

The following rows are synthetic; their statuses are not a verified retailer
taxonomy. The observation key is **retailer + listing ID + capture time**.
The same VIN at different retailers remains separate evidence. Inspect missing
fields and duplicate keys before reading the normalized table.


In [ ]:
observed = pd.read_csv(source_file)
print('SYNTHETIC EXAMPLE ONLY:', source_file)
display(observed)

key = ["retailer", "listing_id", "observed_at_utc"]
duplicate_rows = observed[observed.duplicated(key, keep=False)]
missing_keys = observed[key].isna().any(axis=1)
display(observed.isna().sum().rename("missing_values").to_frame())
display(duplicate_rows)
assert not missing_keys.any(), "Missing observation key: inspect the source."
assert duplicate_rows.empty, "Duplicate observation keys: inspect before analysis."

snapshots = observed.copy()
snapshots["observed_at_utc"] = pd.to_datetime(snapshots["observed_at_utc"], utc=True, errors="raise")
snapshots["asking_price_usd"] = pd.to_numeric(snapshots["asking_price_usd"], errors="raise")
# Parse dates before the second check: equivalent timestamp spellings are one key.
assert not snapshots.duplicated(key).any(), "Duplicate normalized observation keys."
display(snapshots)


### Count the latest synthetic observations

Filter to the latest capture, then group by retailer and native status. A missing
price stays missing; a zero count means no row in this fixture's group. These
counts describe the invented example only. Real absence comparisons additionally
require complete, comparable collections, as shown in [Notebook 20](20_carvana_history_analysis.ipynb).


In [ ]:
latest_capture = snapshots["observed_at_utc"].max()
latest = snapshots[snapshots["observed_at_utc"].eq(latest_capture)].copy()
listing_counts = latest.groupby(["retailer", "native_status"], dropna=False).size().rename("observed_listings").reset_index()
display(latest)
display(listing_counts)
print("Synthetic listing counts only; no sale is inferred from this fixture.")
